<a href="https://colab.research.google.com/github/kimlong77620-lgtm/web-phim-ngan/blob/main/google_colab_en.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Set Up

1. click “Edit” -> "Notebook Settings" -> "Hardware accelerator, GPU" -> Save
<img src="https://z3.ax1x.com/2021/03/30/ciQWWV.png">

2. Click the folder icon on the left, upload your video file, and copy the uploaded video path

<img src="https://z3.ax1x.com/2021/03/30/cilvhq.png">

3. Run the code, enter the pasted video path




check whether GPU works

In [1]:
!nvidia-smi

Thu Mar 12 08:27:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc -V

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


# Install Dependencies

# Run

In [3]:
!git clone --depth=1 https://github.com/YaoFANGUK/video-subtitle-extractor.git

Cloning into 'video-subtitle-extractor'...
remote: Enumerating objects: 193, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (177/177), done.
remote: Total 193 (delta 15), reused 159 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (193/193), 964.97 MiB | 28.02 MiB/s, done.
Resolving deltas: 100% (15/15), done.
Updating files: 100% (162/162), done.


In [9]:
# Chuyển vào đúng thư mục của VSE (Bắt buộc dùng %cd)
%cd /content/video-subtitle-extractor

# Cài đặt các thư viện cần thiết
!pip install -r requirements_gpu.txt

# Cấu hình cài đặt
!echo -e '[DEFAULT]\nInterface = English\nLanguage = en\nMode = fast' > /content/video-subtitle-extractor/settings.ini

# Cài đặt PaddlePaddle bản mới tương thích với Colab (2.6.0)
!pip install paddlepaddle-gpu==2.6.0 -f https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html

/content/video-subtitle-extractor
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements_gpu.txt'
Looking in links: https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 749.8/749.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: opt-einsum
    Found existing installation: opt_einsum 3.4.0
    Uninstalling opt_einsum-3.4.0:
      Successfully uninstalled opt_einsum-3.4.0


In [10]:
# Cài đặt thư viện tạo giao diện
!pip install gradio -q

import gradio as gr
import subprocess
import os
import glob
import shutil

# --- HÀM TAB 1: BÓC SUB ---
def run_vse(video_file, lang, ymin, ymax):
    if video_file is None:
        return "Vui lòng chọn video!", None

    video_path = video_file.name

    # Thêm ymin và ymax vào lệnh nếu có nhập số lớn hơn 0
    cmd = f"python ./backend/main.py --mp4path '{video_path}' --lan '{lang}'"
    if ymin > 0:
        cmd += f" --ymin {int(ymin)}"
    if ymax > 0:
        cmd += f" --ymax {int(ymax)}"

    process = subprocess.run(cmd, shell=True, capture_output=True, text=True)

    if process.returncode != 0:
        return f"Có lỗi xảy ra:\n{process.stderr}", None

    srt_files = glob.glob('**/*.srt', recursive=True)
    if not srt_files:
        return "Không tìm thấy file SRT. Có thể video không có sub.", None

    latest_srt = max(srt_files, key=os.path.getctime)
    return "✅ Bóc sub thành công! Hãy tải file SRT tiếng Trung ở bên dưới.", latest_srt

# --- HÀM TAB 2: ÉP SUB (HARDSUB) ---
def burn_subs(video_file, srt_file):
    if video_file is None or srt_file is None:
        return "Vui lòng tải lên ĐỦ cả Video và file SRT!", None

    output_path = "video_thanh_pham.mp4"

    # Copy file SRT ra thư mục gốc đổi tên ngắn gọn để FFmpeg dễ đọc, tránh lỗi đường dẫn
    shutil.copy(srt_file.name, "phude.srt")
    vid_path = video_file.name

    # Lệnh FFmpeg ép sub cứng (Dùng libx264 chuẩn web, copy nguyên gốc âm thanh)
    cmd = f'ffmpeg -y -i "{vid_path}" -vf "subtitles=phude.srt" -c:v libx264 -crf 23 -preset fast -c:a copy "{output_path}"'

    process = subprocess.run(cmd, shell=True, capture_output=True, text=True)

    if process.returncode != 0:
         return f"❌ Lỗi FFmpeg:\n{process.stderr}", None

    return "✅ Ép phụ đề thành công! Tải video hoàn chỉnh ở bên dưới.", output_path

# --- XÂY DỰNG GIAO DIỆN WEB ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("## 🎬 Trạm Xử Lý Phim Ngắn (Bóc Sub & Ép Sub bằng GPU)")

    with gr.Tabs():
        # Tab 1
        with gr.TabItem("1. Bóc Phụ Đề (VSE)"):
            with gr.Row():
                with gr.Column():
                    vid_input_1 = gr.File(label="Tải video gốc lên (.mp4)", file_types=["video"])
                    lang_input = gr.Dropdown(choices=["ch", "en", "ko", "ja"], value="ch", label="Ngôn ngữ")

                    gr.Markdown("*Mẹo: Nhập tọa độ vùng Sub để bóc nhanh & tránh quét nhầm chữ trên màn hình. Để số 0 nếu muốn quét toàn bộ video.*")
                    with gr.Row():
                        ymin_input = gr.Number(label="Điểm bắt đầu (ymin - pixel)", value=0)
                        ymax_input = gr.Number(label="Điểm kết thúc (ymax - pixel)", value=0)

                    btn_extract = gr.Button("🚀 Bắt đầu bóc chữ", variant="primary")
                with gr.Column():
                    txt_output_1 = gr.Textbox(label="Trạng thái", lines=2)
                    file_output_1 = gr.File(label="Tải file SRT về máy")

            # Đưa ymin và ymax vào input của nút bấm
            btn_extract.click(fn=run_vse, inputs=[vid_input_1, lang_input, ymin_input, ymax_input], outputs=[txt_output_1, file_output_1])

        # Tab 2
        with gr.TabItem("2. Ép Phụ Đề (Hardsub)"):
            gr.Markdown("*Lưu ý: Bạn hãy dịch file SRT sang tiếng Việt trước, sau đó up file SRT tiếng Việt lên đây cùng video gốc.*")
            with gr.Row():
                with gr.Column():
                    vid_input_2 = gr.File(label="Tải video gốc lên (.mp4)", file_types=["video"])
                    srt_input_2 = gr.File(label="Tải file SRT Tiếng Việt lên (.srt)", file_types=[".srt"])
                    btn_burn = gr.Button("🔥 Bắt đầu ép phụ đề", variant="primary")
                with gr.Column():
                    txt_output_2 = gr.Textbox(label="Trạng thái", lines=2)
                    vid_output_2 = gr.File(label="Tải Video thành phẩm")

            btn_burn.click(fn=burn_subs, inputs=[vid_input_2, srt_input_2], outputs=[txt_output_2, vid_output_2])

# Khởi chạy
demo.launch(share=True, debug=True)

/tmp/ipykernel_594/1801017453.py:52: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4d5c8c3d10db4f841c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://4d5c8c3d10db4f841c.gradio.live


Here is an example:

input video path: /content/video-subtitle-extractor/test/test_en.mp4

input subtitle area: 612 717 90 1191

In [8]:
!python ./backend/main.py

Traceback (most recent call last):
  File "/content/video-subtitle-extractor/./backend/main.py", line 16, in <module>
    from Levenshtein import ratio
ModuleNotFoundError: No module named 'Levenshtein'
